# Automatización gráficos Balanza de Pagos

Objetivo: Automatizar en excel los gráficos de Cuenta corriente y Cuenta financiera

Fecha: Octubre 2026

Autor: Miguel Espinal Valencia

### Librerías

In [165]:
#pip install unidecode
import os
import pandas as pd
from unidecode import unidecode
import unicodedata
import re
from pathlib import Path


### Rutas

In [166]:
ruta_principal = "C:\\Users\\espin\\Desktop\\Semillero_coyuntura_economica\\balanza-de-pagos"
ruta_inputs = os.path.join(ruta_principal, "data")
ruta_scripts = os.path.join(ruta_principal, "src")
ruta_outputs = os.path.join(ruta_principal, "output")
ruta_interm = os.path.join(ruta_inputs, "interm")


### Procesamiento de Datos

Resumen Completo

In [167]:
resumen = pd.read_excel(os.path.join(ruta_inputs, "raw\\banrep\\1_Balanza_de_pagos_Resumen.xlsx"))
# Organizo completo los años que están en filas y los junto con los trimestres
resumen.iloc[0] = resumen.iloc[0].ffill()
resumen.iloc[0] = resumen.iloc[0].astype(str).str[:4]
resumen.iloc[0,0] = "Año"
resumen = resumen.dropna().reset_index(drop=True)
resumen = resumen.replace({"Q1":"I", "Q2":"II", "Q3":"III", "Q4":"IV"})
resumen[resumen.columns[0]] = resumen[resumen.columns[0]].str.strip()
resumen = resumen.replace("\n", "")

# Voy a generar dos bases. Una que va a tener toda la informacion incluyendo credito y debito y la otra que no
resumen = resumen.T.reset_index(drop=True)

C:\Users\espin\AppData\Local\Temp\ipykernel_18516\2839649973.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  resumen.iloc[0] = resumen.iloc[0].ffill()


Resumen Para Gráfico

In [168]:
resumen_sin_cb = resumen 

resumen_sin_cb.columns = resumen_sin_cb.iloc[0]
resumen_sin_cb = resumen_sin_cb.drop(0).reset_index(drop=True)

# Conservar solo las columnas de interés
col_interes = ["Año", 
               "Cuentas", 
               "1.A.a Bienes",
               "1.A.b Servicios",
               "1.B Ingreso primario (Renta factorial)",
               "1.C Ingreso secundario (Transferencias corrientes)",
               "3 Cuenta financiera",
               "3.1 Inversión directa",
               "3.2 Inversión de cartera",
               "3.3 Derivados financieros (distintos de reservas) y opciones de compra de acciones por parte de empleados",
               "3.4 Otra inversión",
               "3.5 Activos de reserva",
               "Errores y omisiones netos"
]

resumen_sin_cb = resumen_sin_cb[col_interes]

resumen_sin_cb = resumen_sin_cb.rename(columns={"Cuentas" : "Trimestre", 
               "1.A.a Bienes" : "Bienes",
               "1.A.b Servicios": "Servicios",
               "1.B Ingreso primario (Renta factorial)": "Ingreso primario (Renta factorial)",
               "1.C Ingreso secundario (Transferencias corrientes)": "Ingreso secundario (Transferencias corrientes)",
               "3 Cuenta financiera" : "Cuenta financiera",
               "3.1 Inversión directa" : "Inversión directa",
               "3.2 Inversión de cartera": "Inversión de cartera",
               "3.3 Derivados financieros (distintos de reservas) y opciones de compra de acciones por parte de empleados" : "Derivados financieros",
               "3.4 Otra inversión": "Otra inversión",
               "3.5 Activos de reserva": "Activos de reserva",
               "Errores y omisiones netos": "Errores y omisiones netos"})

resumen_sin_cb = resumen_sin_cb.sort_values(by = ["Año", "Trimestre"]).reset_index(drop=True)

### Datos Cuenta corriente como porcentaje del PIB

In [169]:
CA_pib = pd.read_excel(os.path.join(ruta_inputs, "raw\\banrep\\CA_PIB.xlsx"))
CA_pib = CA_pib.drop(0).reset_index(drop=True)
CA_pib = CA_pib.dropna()
CA_pib["Fecha"] = pd.to_datetime(CA_pib["Fecha"], format="%Y/%m/%d")
CA_pib["Mes"] = CA_pib["Fecha"].dt.month
CA_pib["Año"] = CA_pib["Fecha"].dt.year.astype(str)
CA_pib["Trimestre"] = CA_pib["Mes"].apply(lambda x: "I" if x in [3] else ("II" if x in [6] else ("III" if x in [9] else ("IV" if x in [12] else None))))
CA_pib = CA_pib.rename(columns={"Cuenta corriente, porcentaje del PIB, trimestral": "Cuenta corriente (% PIB)"})
CA_pib = CA_pib.drop(columns=["Fecha", "Mes"])
CA_pib["Cuenta corriente (% PIB)"] = CA_pib["Cuenta corriente (% PIB)"]/100


In [170]:
resumen_ca = resumen_sin_cb[["Año", "Trimestre", "Bienes", "Servicios", "Ingreso primario (Renta factorial)", "Ingreso secundario (Transferencias corrientes)"]]
resumen_ca_pib = pd.merge(resumen_ca, CA_pib, on=["Año", "Trimestre"], how="left")
resumen_ca_pib = resumen_ca_pib.groupby(["Año", "Trimestre"]).sum()

### Cuenta Financiera

In [171]:
resumen_cf = resumen_sin_cb[["Año", "Trimestre", "Cuenta financiera", "Inversión directa", "Inversión de cartera", "Derivados financieros", "Otra inversión", "Activos de reserva"]]

PIB corriente

In [172]:
pib = pd.read_excel(os.path.join(ruta_inputs, "raw\\DANE\\PIB.xlsx"), sheet_name="Cuadro 3", skiprows=11)

# Conservar solo la fila 1 y la 103
pib = pib.iloc[[0, 90]]

pib = pib.drop(columns=pib.columns[[0, 1, 2]]).reset_index(drop=True)

# Renombrar las columnas moviendo los nombres de los años a los unnamed

# Convertir los nombres de columnas Unnamed en valores faltantes
nombres_columnas = pd.Series(pib.columns, dtype="object")

nombres_columnas = nombres_columnas.mask(
    nombres_columnas.astype("string").str.startswith("Unnamed", na=False)
)

# Copiar el año anterior hasta encontrar un nuevo año
pib.columns = nombres_columnas.ffill().to_list()
pib = pib.T.reset_index()
pib.columns = pib.iloc[0]
pib = pib.drop(index=0).reset_index(drop=True)
pib = pib.rename(columns={
    "Concepto": "Año",
    pib.columns[1]: "Trimestre",
    pib.columns[2]: "PIB"
})

pib["Año"] = (
    pib["Año"]
    .astype(str)
    .str.replace("pr", "", regex=False)
)

pib["PIB"] = pd.to_numeric(pib["PIB"], errors="coerce")

In [173]:
tasa_cambio = pd.read_excel(os.path.join(ruta_inputs, "raw\\BANREP\\5_Serie_historica_TRM_mensual_promedio_fin.xlsx"), sheet_name="sheet1")
tasa_cambio = tasa_cambio.rename(columns={"Fecha (dd/mm/aaaa)": "Fecha"})
tasa_cambio["Mes"] = tasa_cambio["Fecha"].dt.month
tasa_cambio["Año"] = tasa_cambio["Fecha"].dt.year.astype(str)

# Promedio trimestral de la tasa de cambio
tasa_cambio["Trimestre"] = tasa_cambio["Mes"].apply(lambda x: "I" if x in [1, 2, 3] else ("II" if x in [4, 5, 6] else ("III" if x in [7, 8, 9] else "IV")))
# Conservar solo las columnas de interés
tasa_cambio = tasa_cambio[["Año", "Trimestre", "Promedio mensual"]]
tasa_cambio = tasa_cambio.groupby(["Año", "Trimestre"]).mean().reset_index()

pib_dolares = pd.merge(pib, tasa_cambio, on=["Año", "Trimestre"], how="left")
pib_dolares["PIB"] = pib_dolares["PIB"]*1000000000
pib_dolares["PIB_dolares"] = pib_dolares["PIB"] / pib_dolares["Promedio mensual"]
pib_dolares = pib_dolares[["Año", "Trimestre", "PIB_dolares"]]
pib_dolares["PIB_dolares"] = pib_dolares["PIB_dolares"] / 1000000

In [182]:
resumen_cf_pib = pd.merge(resumen_cf, pib_dolares, on=["Año", "Trimestre"], how="left")
resumen_cf_pib = resumen_cf_pib.dropna().reset_index(drop=True)
resumen_cf_pib["Cuenta Financiera  (% PIB)"] = resumen_cf_pib["Cuenta financiera"] / resumen_cf_pib["PIB_dolares"]
resumen_cf_pib = resumen_cf_pib.drop(columns=["Cuenta financiera", "PIB_dolares"])
resumen_cf_pib = resumen_cf_pib.groupby(["Año", "Trimestre"]).sum()

In [183]:
with pd.ExcelWriter(os.path.join(ruta_outputs, "Graficos.xlsx"), engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    resumen_ca_pib.to_excel(writer, sheet_name="Resumen CA", index=True)
    resumen_cf_pib.to_excel(writer, sheet_name="Resumen CF", index=True)

In [184]:
with pd.ExcelWriter(os.path.join(ruta_interm, "Resumen_completo.xlsx"), engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    resumen.to_excel(writer, sheet_name="Resumen", index=True)